In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. LOAD DATA
df = pd.read_csv("medical_insurance.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# 3. BASIC DATA ANALYSIS
print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes.value_counts())

# 4. PREPROCESSING - HANDLE MISSING VALUES
print("\nnote: Outcome from previous commit gave missing values in dataset analysis. Handling missing values in the next step before encoding categorical features.")

# Check alcohol_freq values before fixing
print("\nAlcohol frequency before handling:")
print(df['alcohol_freq'].value_counts(dropna=False))

# Fill NaN with 'Unknown' as a valid category
df['alcohol_freq'] = df['alcohol_freq'].fillna('Unknown')

print("\nAlcohol frequency after handling:")
print(df['alcohol_freq'].value_counts())

# Verify no missing values remain
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

# 5. DROP UNNECESSARY FEATURES
columns_to_drop = [
    "person_id",
    "policy_term_years",
    "policy_changes_last_2yrs",
    "provider_quality",
    "risk_score",
    "annual_premium",
    "monthly_premium",
    "claims_count",
    "avg_claim_amount",
    "total_claims_paid"
]

df = df.drop(columns=columns_to_drop)

print("\nRemaining Features:", df.shape[1])

# 6. ENCODING CATEGORICAL DATA
categorical_cols = df.select_dtypes(include=["object"]).columns
print(f"\nCategorical columns to encode: {categorical_cols.tolist()}")

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"Encoded {col}: {len(le.classes_)} unique values")

# 7. DEFINE FEATURES & TARGET
X = df.drop("annual_medical_cost", axis=1)
y = df["annual_medical_cost"]

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# 8. DATA SPLITTING (70 / 10 / 20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=42
)

print("\nData split completed.")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# 9. SETTING HYPERPARAMETERS
hyperparams = {
    'n_estimators': 100,      # Number of boosting stages
    'learning_rate': 0.1,     # Shrinks contribution of each tree
    'max_depth': 3,           # Maximum depth of individual trees
    'min_samples_split': 2,   # Minimum samples required to split node
    'min_samples_leaf': 1,    # Minimum samples required at leaf node
    'subsample': 1.0,         # Fraction of samples used for fitting trees
    'random_state': 42        # For reproducibility
}


Dataset Shape: (100000, 54)

First 5 rows:
   person_id  age     sex   region urban_rural  income     education  \
0      75722   52  Female    North    Suburban   22700     Doctorate   
1      80185   79  Female    North       Urban   12800         No HS   
2      19865   68    Male    North       Rural   40700            HS   
3      76700   15    Male    North    Suburban   15600  Some College   
4      92992   53    Male  Central    Suburban   89600     Doctorate   

  marital_status employment_status  household_size  ...  liver_disease  \
0        Married           Retired               3  ...              0   
1        Married          Employed               3  ...              0   
2        Married           Retired               5  ...              0   
3        Married     Self-employed               5  ...              0   
4        Married     Self-employed               2  ...              0   

   arthritis mental_health proc_imaging_count  proc_surgery_count  \
0         